In [5]:
#imports
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# --- Dataset class ---
class SpectrogramDataset(Dataset):
    def __init__(self, folder_path, txt_file):
        self.folder_path = folder_path
        self.samples = []
        with open(txt_file, "r") as f:
            for line in f:
                file_path, label = line.strip().split()
                full_path = os.path.join(folder_path, file_path)
                label = int(label)
                self.samples.append((full_path, label))
                
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        data = np.load(file_path)
        
        # Normalize each sample (min-max)
        data = (data - data.min()) / (data.max() - data.min() + 1e-6)
        
        data_tensor = torch.tensor(data, dtype=torch.float32).unsqueeze(0)  # add channel dim
        return data_tensor, label

# --- CNN model with dynamic flatten size ---
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=9, input_shape=(1, 64, 64)):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        
        with torch.no_grad():
            x = torch.zeros(1, *input_shape)
            x = self.pool(torch.relu(self.bn1(self.conv1(x))))
            x = self.pool(torch.relu(self.bn2(self.conv2(x))))
            x = self.pool(torch.relu(self.bn3(self.conv3(x))))
            flatten_size = x.numel()
        
        self.fc1 = nn.Linear(flatten_size, 128)
        self.fc2 = nn.Linear(128, num_classes)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# --- Training function ---
def train_model():
    folder_path = "/anvil/projects/x-cis220051/corporate/aerospace-rf/fiot_highway2-main"
    train_txt = os.path.join(folder_path, "train.txt")
    batch_size = 32
    epochs = 10
    learning_rate = 0.001
    num_classes = 9
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    dataset = SpectrogramDataset(folder_path, train_txt)
    dataset_size = len(dataset)
    
    if dataset_size == 0:
        print("Dataset is empty! Check file paths.")
        return
    
    # Get input shape from first sample
    sample_data, _ = dataset[0]
    input_shape = sample_data.shape  # e.g. (1, 64, 64)
    print(f"Input data shape: {input_shape}")
    
    train_size = int(0.8 * dataset_size)
    val_size = dataset_size - train_size
    
    train_ds, val_ds = random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    
    model = SimpleCNN(num_classes=num_classes, input_shape=input_shape).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = running_loss / train_size
        train_acc = correct / total
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        val_loss /= val_size
        val_acc = val_correct / val_total
        
        print(f"Epoch {epoch+1}/{epochs} — Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    print("Training complete.")

if __name__ == "__main__":
    train_model()

Input data shape: torch.Size([1, 512, 243])


Epoch 1/10 [Train]: 100%|██████████| 323/323 [13:14<00:00,  2.46s/it]


Epoch 1/10 — Train Loss: 2.8961, Train Acc: 0.5355, Val Loss: 0.9931, Val Acc: 0.6365


Epoch 2/10 [Train]: 100%|██████████| 323/323 [13:16<00:00,  2.47s/it]


Epoch 2/10 — Train Loss: 1.1665, Train Acc: 0.5619, Val Loss: 0.9569, Val Acc: 0.6450


...

Epoch 8/10 — Train Loss: 1.0599, Train Acc: 0.6000, Val Loss: 1.0175, Val Acc: 0.6241


Epoch 9/10 [Train]: 100%|██████████| 323/323 [13:10<00:00,  2.45s/it]


Epoch 9/10 — Train Loss: 1.0533, Train Acc: 0.5971, Val Loss: 1.0309, Val Acc: 0.6098


Epoch 10/10 [Train]: 100%|██████████| 323/323 [13:10<00:00,  2.45s/it]


Epoch 10/10 — Train Loss: 1.0712, Train Acc: 0.5962, Val Loss: 0.9027, Val Acc: 0.6554
Training complete.
